# Glacier Facies Classification via Unsupervised Clustering of Sentinel-1 Coherence
### Case study: Aletsch Glacier, Swiss Alps

This notebook implements **use case 3**: an **unsupervised machine-learning** classification
of glacier surface facies (dry snow / wet snow-firn / bare ice) and a melt-onset proxy, built
from a short-baseline Sentinel-1 coherence stack via
[`sentinel1_sar_coherence`](https://algorithm-catalogue.apex.esa.int/apps/sentinel1_sar_coherence)
on the Copernicus Data Space Ecosystem (CDSE) openEO back-end — no phase unwrapping,
displacement estimation, or time-series inversion involved.

**Why coherence for glacier facies?** Different glacier surface states decorrelate at very
different rates: dry snow stays coherent for weeks, the percolation/wet-snow zone
decorrelates quickly once meltwater appears, and bare ice sits in between with its own
characteristic backscatter and coherence behaviour. A short coherence + backscatter stack
spanning the winter→melt-season transition therefore carries enough structure to separate
these classes **without labelled training data** — a natural fit for fast, unsupervised
`sklearn` clustering.

**Area of interest:** Aletsch Glacier, Switzerland — the largest glacier in the Alps, well
within Europe and extensively studied.

**What this notebook does:**
1. Discovers a Sentinel-1 burst over the glacier.
2. Builds a short-baseline coherence stack spanning the winter→melt transition.
3. Applies a **custom UDF** that builds a per-pixel feature vector (coherence mean/trend,
   backscatter mean/std) and clusters it with `sklearn.cluster.MiniBatchKMeans` (fast, scales
   to millions of pixels) — `sklearn.mixture.GaussianMixture` is noted as a soft-clustering
   alternative for gradational facies boundaries.
4. Visualises the resulting facies map and a melt-onset proxy.


In [1]:
# --- Imports ---
import requests
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import openeo


In [2]:
# --- Connect to the openEO back-end on CDSE ---
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()


Authenticated using refresh token.


## 1. Area of interest

A box covering the accumulation and ablation zones of the Aletsch Glacier.

In [3]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [7.95, 46.43],
            [7.95, 46.58],
            [8.15, 46.58],
            [8.15, 46.43],
            [7.95, 46.43],
        ]
    ],
}


## 2. Find a Sentinel-1 burst covering the AOI

In [4]:
def find_candidate_bursts(aoi_polygon, start, end, top=20):
    coords = aoi_polygon["coordinates"][0]
    wkt_coords = ", ".join(f"{lon} {lat}" for lon, lat in coords)
    footprint_wkt = f"POLYGON(({wkt_coords}))"

    filter_str = (
        f"OData.CSC.Intersects(area=geography'SRID=4326;{footprint_wkt}') "
        f"and ContentDate/Start gt {start}T00:00:00.000Z "
        f"and ContentDate/Start lt {end}T00:00:00.000Z "
        f"and PolarisationChannels eq 'VV'"
    )
    url = (
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Bursts"
        f"?$filter={filter_str}&$top={top}&$orderby=ContentDate/Start asc"
    )
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json().get("value", [])


candidates = find_candidate_bursts(aoi, "2023-03-01", "2023-03-15")
for b in candidates:
    print(
        b.get("Id"), "|",
        "burst_id:", b.get("BurstId"),
        "swath:", b.get("SwathIdentifier"),
        "orbit:", b.get("RelativeOrbitNumber"),
        "direction:", b.get("OrbitDirection"),
    )


adf63ed1-a718-44ff-99c0-f526f5db1b71 | burst_id: 297211 swath: IW1 orbit: 139 direction: DESCENDING
38c00768-3da7-4204-ae33-32e795b1a53d | burst_id: 297212 swath: IW1 orbit: 139 direction: DESCENDING
0e824bd8-2c8b-4c00-a6cd-1527353b5ede | burst_id: 30345 swath: IW1 orbit: 15 direction: ASCENDING
fcf2e6b9-f893-4153-8b82-94b98aae5d61 | burst_id: 30346 swath: IW1 orbit: 15 direction: ASCENDING
8d9cbee0-f841-41fa-8db1-524a449d898a | burst_id: 30347 swath: IW1 orbit: 15 direction: ASCENDING
70a59a28-1031-4609-b26c-1050b1b6837d | burst_id: 140413 swath: IW2 orbit: 66 direction: DESCENDING
e6f2dcf6-2fb9-4419-84b8-ddc779aa7948 | burst_id: 140413 swath: IW3 orbit: 66 direction: DESCENDING
5ef66abe-c5f0-40d0-8fa4-6be70050d994 | burst_id: 140414 swath: IW2 orbit: 66 direction: DESCENDING
a4382b68-5d4b-497d-927d-d72264f04e79 | burst_id: 187142 swath: IW3 orbit: 88 direction: ASCENDING
369d2211-761c-4111-9238-66c5a920368f | burst_id: 187143 swath: IW3 orbit: 88 direction: ASCENDING
2954e269-b4a4-4f

In [5]:
BURST_ID = 140414           # <-- replace with a real burst_id for the AOI
SUB_SWATH = "IW2"          # <-- replace as needed

# Span the winter (stable, dry snow) through peak melt season transition
STACK_START = "2023-03-01"
STACK_END = "2023-07-31"
TEMPORAL_BASELINE_DAYS = 12


## 3. Build the coherence stack

Same `temporal_extent` + `temporal_baseline` pattern as before — a fixed short-baseline
consecutive-pair coherence series, not a redundant network.

In [6]:
coherence_stack = connection.datacube_from_process(
    "sentinel1_sar_coherence",
    namespace=(
        "https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/"
        "algorithm_catalog/eurac/sentinel1_sar_coherence/openeo_udp/"
        "sentinel1_sar_coherence.json"
    ),
    **{
        "temporal_extent": [STACK_START, STACK_END],
        "temporal_baseline": TEMPORAL_BASELINE_DAYS,
        "burst_id": BURST_ID,
        "coherence_window_az": 2,
        "coherence_window_rg": 10,
        "polarization": "VV",
        "sub_swath": SUB_SWATH,
    },
)


Check the actual bands returned (`print(coherence_stack.metadata)`) before running the
UDF below — depending on the UDP version, the coherence product may also carry a backscatter
amplitude band alongside `coherence`. The UDF assumes bands `"coherence"` and, if present,
`"amplitude"`; if only coherence is available, drop the amplitude-derived features (the code
below is written to degrade gracefully if `"amplitude"` is missing).

## 4. UDF: per-pixel feature vector + fast unsupervised clustering

Feature vector per pixel, built from the time series:
- mean coherence
- coherence linear trend (slope over the stack — captures the seasonal decorrelation onset)
- backscatter mean / std, if available

Clustering: `MiniBatchKMeans` — O(n), scales to millions of pixels, and is fit directly on
each processed chunk here for a self-contained demo. For consistent cluster labels across
many tiles in a large-scale job, fit `MiniBatchKMeans` once on a representative sample and
pass the fitted cluster centroids into the UDF via `context`, then call `.predict()` only
inside the per-tile UDF (sketched at the bottom of this section).

`GaussianMixture` is noted as an alternative when a soft/probabilistic assignment near facies
boundaries (e.g. around the equilibrium line) is preferred over a hard partition.


In [7]:
glacier_facies_udf = """
# /// script
# dependencies = ["scikit-learn"]
# ///
import numpy as np
import xarray as xr
from sklearn.cluster import MiniBatchKMeans

N_CLUSTERS = 3  # e.g. dry snow / wet snow-firn / bare ice


def apply_datacube(cube: xr.DataArray, context: dict) -> xr.DataArray:
    coherence = cube.sel(bands="coherence")

    has_amplitude = "amplitude" in cube["bands"].values
    amplitude = cube.sel(bands="amplitude") if has_amplitude else None

    t_index = np.arange(coherence.sizes["t"])

    coh_mean = coherence.mean(dim="t", skipna=True)
    # linear trend of coherence over the stack (per-pixel polyfit, vectorised)
    coh_values = coherence.values.reshape(coherence.sizes["t"], -1)
    valid = ~np.isnan(coh_values).all(axis=0)
    coh_trend_flat = np.full(coh_values.shape[1], np.nan)
    if valid.any():
        y = np.where(np.isnan(coh_values[:, valid]), 0, coh_values[:, valid])
        A = np.vstack([t_index, np.ones_like(t_index)]).T
        slopes, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        coh_trend_flat[valid] = slopes[0]
    coh_trend = coh_mean.copy(data=coh_trend_flat.reshape(coh_mean.shape))

    feature_list = [coh_mean.values.ravel(), coh_trend.values.ravel()]
    if has_amplitude:
        feature_list.append(amplitude.mean(dim="t", skipna=True).values.ravel())
        feature_list.append(amplitude.std(dim="t", skipna=True).values.ravel())

    features = np.column_stack(feature_list)
    valid_mask = ~np.isnan(features).any(axis=1)

    labels = np.full(features.shape[0], -1, dtype=float)
    if valid_mask.sum() >= N_CLUSTERS:
        km = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=0, n_init="auto")
        labels[valid_mask] = km.fit_predict(features[valid_mask]).astype(float)

        # relabel clusters by ascending mean coherence, so label 0 = most-decorrelated
        # (typically wet/melt-affected) and the highest label = most stable (dry snow)
        order = np.argsort(
            [features[valid_mask][labels[valid_mask] == k, 0].mean() for k in range(N_CLUSTERS)]
        )
        remap = {old: new for new, old in enumerate(order)}
        labels[valid_mask] = np.vectorize(remap.get)(labels[valid_mask])

    facies = coh_mean.copy(data=labels.reshape(coh_mean.shape))

    result = xr.concat([facies, coh_mean, coh_trend], dim="bands")
    result = result.assign_coords(bands=["facies_cluster", "coherence_mean", "coherence_trend"])
    return result
"""


In [8]:
s1_glacier_facies = coherence_stack.apply_dimension(
    code=glacier_facies_udf,
    runtime="Python",
    dimension="t",
)


C:\Users\SHARMAP\AppData\Local\Temp\ipykernel_8960\139355527.py:1: UserDeprecationWarning: Specifying UDF code through `code`, `runtime` and `version` arguments is deprecated. Instead create an `openeo.UDF` object and pass that to the `process` argument.
  s1_glacier_facies = coherence_stack.apply_dimension(


> `MiniBatchKMeans` fit-once / predict-many sketch for large-scale, tile-consistent runs
> (not executed here):
> ```python
> # 1. run a small aggregate_spatial / low-res pass first to get a representative sample
> # 2. fit centroids = MiniBatchKMeans(...).fit(sample_features).cluster_centers_
> # 3. json.dumps(centroids.tolist()) and pass via .apply_dimension(..., context={"centroids": ...})
> # 4. inside apply_datacube, read context["centroids"] and call cdist/argmin instead of .fit_predict
> ```

## 5. Execute the batch job

In [ ]:
job = s1_glacier_facies.create_job(
    title="aletsch_glacier_facies_clustering",
    outputformat="netCDF",
)
job.start_and_wait()
job.get_results().download_files("aletsch_glacier_facies")


0:00:00 Job 'j-260728090834494f845cd2f79555c1b3': send 'start'
0:00:02 Job 'j-260728090834494f845cd2f79555c1b3': created (progress 0%)
0:00:08 Job 'j-260728090834494f845cd2f79555c1b3': queued (progress 0%)
0:00:14 Job 'j-260728090834494f845cd2f79555c1b3': queued (progress 0%)
0:00:22 Job 'j-260728090834494f845cd2f79555c1b3': queued (progress 0%)
0:00:32 Job 'j-260728090834494f845cd2f79555c1b3': queued (progress 0%)
0:00:44 Job 'j-260728090834494f845cd2f79555c1b3': queued (progress 0%)
0:01:00 Job 'j-260728090834494f845cd2f79555c1b3': running (progress N/A)
0:01:19 Job 'j-260728090834494f845cd2f79555c1b3': running (progress N/A)
0:01:43 Job 'j-260728090834494f845cd2f79555c1b3': running (progress N/A)
0:02:13 Job 'j-260728090834494f845cd2f79555c1b3': running (progress N/A)
0:02:51 Job 'j-260728090834494f845cd2f79555c1b3': running (progress N/A)
0:03:37 Job 'j-260728090834494f845cd2f79555c1b3': running (progress N/A)
0:04:36 Job 'j-260728090834494f845cd2f79555c1b3': running (progress N/A)

## 6. Plot the facies map

In [ ]:
result_ds = xr.load_dataset("aletsch_glacier_facies/openEO.nc")

facies = result_ds["facies_cluster"] if "facies_cluster" in result_ds else (
    result_ds.to_array(dim="bands").sel(bands="facies_cluster")
)

n_clusters = int(np.nanmax(facies.values)) + 1
labels = ["Melt-affected / wet snow", "Bare ice / transitional", "Dry snow / stable"][:n_clusters]
colors = ["#08519c", "#6baed6", "#ffffff"][:n_clusters]

cmap = plt.matplotlib.colors.ListedColormap(colors)

fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
facies.squeeze().plot.imshow(ax=ax, cmap=cmap, vmin=-0.5, vmax=n_clusters - 0.5, add_colorbar=False)
ax.set_title("Aletsch Glacier — unsupervised facies clustering")
ax.set_xlabel("")
ax.set_ylabel("")

patches = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(n_clusters)]
fig.legend(handles=patches, bbox_to_anchor=(0.95, 0.3), loc=1)
plt.tight_layout()
plt.show()


## Notes & limitations

- `MiniBatchKMeans` is fit independently per processed chunk in this demo version, so exact
  cluster *ordering* is stabilised by the coherence-based relabeling step, but boundaries
  between adjacent tiles may not be perfectly consistent — see the fit-once/predict-many
  sketch above for a production-grade version.
- `GaussianMixture` (`sklearn.mixture.GaussianMixture`) is a straightforward drop-in
  replacement for `MiniBatchKMeans` if you want soft cluster probabilities instead of a hard
  partition, at extra compute cost — worth it near the equilibrium line where facies
  transitions are gradual rather than sharp.
- Avoid `DBSCAN` / `HDBSCAN` / agglomerative clustering directly on all pixels of a tile —
  their O(n²) memory/time cost makes them impractical inside a per-chunk UDF at this scale.
- No phase unwrapping, displacement, or time-series inversion is used anywhere in this
  workflow — only pairwise coherence statistics and unsupervised clustering.
- A natural extension (not implemented here, to keep this notebook to a single, self-contained
  clustering pass) is combining the facies raster with the Copernicus DEM to derive an
  equilibrium-line-altitude (ELA) proxy per season.
